# N085 · Bellman-Ford、受限路径与Floyd

**目标：** 区分限制边数的路径和一般最短路径。

**先修：** N084, N031。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 分轮松弛；上一轮副本；负边/负环；全源DP；中间点顺序。

## 从问题到算法

Bellman-Ford把“允许使用多少条边”当作逐轮放宽的限制。若第n轮仍可从源可达地改进，存在可达负环，此时不能把当前数值当有限最短路。限制中转的航班题每轮必须读取上一轮快照，禁止一次轮次内连用多条新边。Floyd则逐个放开中间顶点。

## 最小实现

**本章接口：** `bellman_ford(n, edges, source)`、`cheapest_flights(n, flights, src, dst, k)`、`floyd_warshall(n, edges)`、`find_the_city(n, edges, threshold)`

In [1]:
def bellman_ford(n,edges,source):
    distance=[float('inf')]*n; distance[source]=0
    for _ in range(n-1):
        changed=False
        for u,v,w in edges:
            if distance[u]+w<distance[v]: distance[v]=distance[u]+w; changed=True
        if not changed: break
    negative_cycle=any(distance[u]+w<distance[v] for u,v,w in edges)
    return distance,negative_cycle

def cheapest_flights(n,flights,src,dst,k):
    distance=[float('inf')]*n; distance[src]=0
    for _ in range(k+1):
        new=distance[:]
        for u,v,w in flights: new[v]=min(new[v],distance[u]+w)
        distance=new
    return -1 if distance[dst]==float('inf') else distance[dst]

def floyd_warshall(n,edges,directed=True):
    distance=[[float('inf')]*n for _ in range(n)]
    for i in range(n): distance[i][i]=0
    for u,v,w in edges:
        distance[u][v]=min(distance[u][v],w)
        if not directed: distance[v][u]=min(distance[v][u],w)
    for k in range(n):
        for i in range(n):
            if distance[i][k]==float('inf'): continue
            for j in range(n): distance[i][j]=min(distance[i][j],distance[i][k]+distance[k][j])
    return distance

def find_the_city(n,edges,threshold):
    distance=floyd_warshall(n,edges,False)
    return min(range(n),key=lambda i:(sum(i!=j and distance[i][j]<=threshold for j in range(n)),-i))

## 正确性、前提与复杂度

无负环时最短简单路径最多n−1条边，因此足够松弛后稳定。Floyd最优路径要么不经k，要么分为i到k和k到j两段；外层k不能随意与i/j调换。

**代价：** Bellman-Ford O(VE)、O(V)空间；限边路径O((k+1)E)、O(V)空间；Floyd O(V³)时间/O(V²)空间。负环时Floyd对角线可用于检测，但本函数不标记所有负无穷影响对。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

flights=[(0,1,100),(1,2,100),(0,2,500)]
show_table(['最多中转k','最多边数','0到2最低票价'],[(k,k+1,cheapest_flights(3,flights,0,2,k)) for k in range(3)])

最多中转k,最多边数,0到2最低票价
0,1,500
1,2,200
2,3,200


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
f=[(0,1,100),(1,2,100),(0,2,500)]
assert cheapest_flights(3,f,0,2,0)==500 and cheapest_flights(3,f,0,2,1)==200
assert cheapest_flights(3,f,2,0,2)==-1
assert floyd_warshall(2,[(0,1,5),(0,1,2)])[0][1]==2
assert bellman_ford(3,[(0,1,1),(1,2,-3),(2,1,1)],0)[1]
assert not bellman_ford(3,[(1,2,-3),(2,1,1)],0)[1]
assert find_the_city(4,[(0,1,3),(1,2,1),(1,3,4),(2,3,1)],4)==3
from random import Random
rng=Random(85)
for _ in range(80):
    n=5; edges=[(u,v,rng.randrange(6)) for u in range(n) for v in range(n) if rng.randrange(4)==0]
    allpairs=floyd_warshall(n,edges)
    for source in range(n):
        dist,negative=bellman_ford(n,edges,source)
        assert not negative and dist==allpairs[source]
print('N085: 所有本章断言通过')

N085: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 构造原地松弛违反K站限制的例子。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 对自编图加入可达负环检测。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 787. Cheapest Flights Within K Stops（canonical）
- 1334. Find the City With the Smallest Number of Neighbors at a Threshold Distance（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。